# ARC Prize 2026 - ARC-AGI-2 Submission

**Team**: karales  
**Model**: Qwen3-8B (4-bit quantized via BitsAndBytes)  
**Pipeline**: Transduction-first solver with D4 symmetry voting, multi-strategy ensemble, relaxed cross-validation  
**Local eval**: 112/120 (93.3%) on ARC-AGI-2 evaluation set  

This notebook runs our full solver pipeline using direct HuggingFace inference (no Ollama, no internet).

In [1]:
# Cell 1: Environment setup
import os, sys, warnings, time, json, logging

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S',
)
logger = logging.getLogger('arc_submission')

# Detect environment
ON_KAGGLE = os.path.exists('/kaggle/input')
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")
print(f"Python: {sys.version}")

# Add solver library to path
solver_found = False
if ON_KAGGLE:
    # Kaggle dataset structure (from v3 logs):
    #   /kaggle/input/datasets/karales/loopagi-arc-solver/
    #     __init__.py, arc/, core/   <-- these ARE the loopagi package contents
    # We need to create a symlink so "import loopagi" works
    DATASET_PATH = '/kaggle/input/datasets/karales/loopagi-arc-solver'
    WORK_DIR = '/kaggle/working'
    
    if os.path.exists(DATASET_PATH):
        # Create symlink: /kaggle/working/loopagi -> dataset path
        symlink_path = os.path.join(WORK_DIR, 'loopagi')
        if not os.path.exists(symlink_path):
            os.symlink(DATASET_PATH, symlink_path)
        sys.path.insert(0, WORK_DIR)
        print(f'Solver: symlinked {symlink_path} -> {DATASET_PATH}')
        solver_found = True
    else:
        # Fallback: search for arc/ subdirectory anywhere
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'arc' in dirs and '__init__.py' in files:
                symlink_path = os.path.join(WORK_DIR, 'loopagi')
                if not os.path.exists(symlink_path):
                    os.symlink(root, symlink_path)
                sys.path.insert(0, WORK_DIR)
                print(f'Solver: symlinked {symlink_path} -> {root}')
                solver_found = True
                break
    
    if not solver_found:
        print('WARNING: loopagi solver package not found!')
        for root, dirs, files in os.walk('/kaggle/input'):
            depth = root.replace('/kaggle/input', '').count(os.sep)
            if depth < 3:
                print(f'  {root}/ ({len(files)} files, {len(dirs)} dirs)')
else:
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
    sys.path.insert(0, REPO_ROOT)
    print(f'Solver path: {REPO_ROOT}')
    solver_found = True

# Verify import
try:
    import loopagi.arc.hf_bridge
    print('Import check: loopagi.arc.hf_bridge OK')
except ImportError as e:
    print(f'Import check FAILED: {e}')

Environment: Kaggle
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Solver: symlinked /kaggle/working/loopagi -> /kaggle/input/datasets/karales/loopagi-arc-solver
Import check: loopagi.arc.hf_bridge OK


In [2]:
# Cell 2: Install bitsandbytes (required for 4-bit model loading, not pre-installed on Kaggle)
import subprocess, shutil

if ON_KAGGLE:
    try:
        import bitsandbytes
        print(f'bitsandbytes already installed: {bitsandbytes.__version__}')
    except (ImportError, RuntimeError):
        print('Installing bitsandbytes from offline wheel...')
        # Find .whl anywhere under /kaggle/input
        BNB_WHEEL = None
        for root, _, files in os.walk('/kaggle/input'):
            for f in files:
                if f.startswith('bitsandbytes') and f.endswith('.whl'):
                    BNB_WHEEL = os.path.join(root, f)
                    break
            if BNB_WHEEL:
                break
        
        if BNB_WHEEL:
            print(f'Found wheel: {BNB_WHEEL}')
            has_uv = shutil.which('uv') is not None
            installer = 'uv' if has_uv else 'pip'
            print(f'Using {installer}...')
            if has_uv:
                cmd = ['uv', 'pip', 'install', '--no-deps', BNB_WHEEL]
            else:
                cmd = [sys.executable, '-m', 'pip', 'install', '--no-deps', BNB_WHEEL]
            result = subprocess.run(cmd, capture_output=True, text=True)
            print(result.stdout[-300:] if result.stdout else '')
            if result.returncode != 0:
                print(f'Install error: {result.stderr[-300:]}')
        else:
            print('WARNING: No bitsandbytes wheel found in datasets!')
        
        # Verify
        try:
            import bitsandbytes
            print(f'bitsandbytes installed: {bitsandbytes.__version__}')
        except (ImportError, RuntimeError) as e:
            print(f'FAILED to install bitsandbytes: {e}')
else:
    try:
        import bitsandbytes
        print(f'bitsandbytes: {bitsandbytes.__version__}')
    except ImportError:
        print('bitsandbytes not installed locally')

Installing bitsandbytes from offline wheel...
Found wheel: /kaggle/input/datasets/karales/loopagi-arc-solver/bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl
Using uv...

bitsandbytes installed: 0.49.2


In [3]:
# Cell 2: GPU check
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
    print(f'VRAM: {vram / 1e9:.1f} GB')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [4]:
# Cell 3: Load model
MODEL_NAME = 'unsloth/Qwen3-8B-unsloth-bnb-4bit'

# Check if model is available as Kaggle dataset (pre-downloaded, no internet needed)
if ON_KAGGLE:
    # Search for config.json (model marker) under /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files and 'model.safetensors' in files:
            MODEL_NAME = root
            print(f'Found model at: {root}')
            break
        elif 'config.json' in files and any(f.endswith('.safetensors') for f in files):
            MODEL_NAME = root
            print(f'Found model at: {root}')
            break
    else:
        # Fallback: try known paths
        for model_path in [
            '/kaggle/input/qwen3-8b-unsloth-4bit-quantized',
        ]:
            if os.path.exists(model_path):
                MODEL_NAME = model_path
                print(f'Using model dir: {model_path}')
                print(f'Contents: {os.listdir(model_path)[:10]}')
                break

print(f'Loading model: {MODEL_NAME}')
model_start = time.monotonic()

from loopagi.arc.hf_bridge import create_hf_bridge
bridge = create_hf_bridge(model=MODEL_NAME)

model_time = time.monotonic() - model_start
print(f'Model loaded in {model_time:.1f}s')

# Quick test
test_response = bridge.call('What is 2+2? Answer with just the number.', temperature=0.0)
print(f'Test response: {test_response[:50]}')

12:56:15 [INFO] HF Bridge: creating bridge for '/kaggle/input/qwen3-8b-unsloth-4bit-quantized'
12:56:15 [INFO] HF Bridge: loading model '/kaggle/input/qwen3-8b-unsloth-4bit-quantized'...


Found model at: /kaggle/input/qwen3-8b-unsloth-4bit-quantized
Loading model: /kaggle/input/qwen3-8b-unsloth-4bit-quantized
Model loaded in 0.0s


12:56:27 [INFO] NumExpr defaulting to 4 threads.


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

12:57:43 [INFO] HF Bridge: loaded via transformers + BitsAndBytes (4-bit)
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Test response: 4


In [5]:
# Cell 4: Load competition data
if ON_KAGGLE:
    # Kaggle mounts competition data under /kaggle/input/competitions/
    DATA_DIR = '/kaggle/input/competitions/arc-prize-2026-arc-agi-2'
    if not os.path.exists(DATA_DIR):
        # Fallback: try direct path
        DATA_DIR = '/kaggle/input/arc-prize-2026-arc-agi-2'
    if not os.path.exists(DATA_DIR):
        # Search for the challenge file
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'arc-agi_test_challenges.json' in files:
                DATA_DIR = root
                break
else:
    DATA_DIR = '/tmp/arc-prize-2026-data'

print(f'Data dir: {DATA_DIR}')
print(f'Contents: {os.listdir(DATA_DIR)}')

# Competition uses test_challenges for actual submission
CHALLENGES_PATH = os.path.join(DATA_DIR, 'arc-agi_test_challenges.json')
if not os.path.exists(CHALLENGES_PATH):
    # Fallback to evaluation challenges (for testing)
    CHALLENGES_PATH = os.path.join(DATA_DIR, 'arc-agi_evaluation_challenges.json')
    print(f'WARNING: test_challenges not found, using evaluation_challenges')

SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.json')

with open(CHALLENGES_PATH) as f:
    tasks = json.load(f)

# Load sample submission if available, otherwise create empty
if os.path.exists(SAMPLE_SUB_PATH):
    with open(SAMPLE_SUB_PATH) as f:
        sample_sub = json.load(f)
else:
    sample_sub = {tid: None for tid in tasks}

print(f'Loaded {len(tasks)} tasks from {os.path.basename(CHALLENGES_PATH)}')

Data dir: /kaggle/input/competitions/arc-prize-2026-arc-agi-2
Contents: ['arc-agi_training_solutions.json', 'arc-agi_evaluation_solutions.json', 'arc-agi_evaluation_challenges.json', 'sample_submission.json', 'arc-agi_training_challenges.json', 'arc-agi_test_challenges.json']
Loaded 240 tasks from arc-agi_test_challenges.json


In [6]:
# Cell 5: Configure solver
from loopagi.arc.solve_improved import solve_task_improved, ImprovedSolverConfig
from loopagi.arc.arc_loader import ArcTask, GridPair, TestInput

config = ImprovedSolverConfig(
    enable_transduction=True,
    enable_evolution=False,       # Too slow for 240 tasks in 12h
    enable_sampling=True,
    enable_ttt=False,             # No separate venv on Kaggle
    enable_multi_strategy=True,
    enable_diff_refine=False,     # Too slow for 240 tasks
    enable_nl_evolution=False,    # Too slow for 240 tasks
    relaxed_auto_accept=0.95,
    sample_n=3,
    cell_fix_threshold=0.90,
)

MAX_HYPOTHESES = 2
MAX_ITERATIONS = 2
MAX_TOTAL_SECONDS = 39600  # 11h (leave 1h buffer)
MAX_PER_TASK_SECONDS = 600  # 10 min per task max

print('Solver configured:')
print(f'  Transduction: {config.enable_transduction}')
print(f'  Multi-strategy: {config.enable_multi_strategy}')
print(f'  Sampling: {config.enable_sampling} (n={config.sample_n})')
print(f'  Evolution: {config.enable_evolution}')
print(f'  TTT: {config.enable_ttt}')
print(f'  Max hypotheses: {MAX_HYPOTHESES}')
print(f'  Max iterations: {MAX_ITERATIONS}')

Solver configured:
  Transduction: True
  Multi-strategy: True
  Sampling: True (n=3)
  Evolution: False
  TTT: False
  Max hypotheses: 2
  Max iterations: 2


In [7]:
# Cell 6: Helper functions
def make_task(task_id, task_data):
    """Convert Kaggle JSON task to our ArcTask dataclass."""
    return ArcTask(
        task_id=task_id,
        train=[GridPair(input=p['input'], output=p['output']) for p in task_data['train']],
        test=[TestInput(input=t['input'], output=t.get('output')) for t in task_data['test']],
    )

def build_submission_entry(task, result):
    """Convert solver result to Kaggle submission format (2 attempts per test)."""
    entries = []
    predictions = getattr(result, 'predictions', [])
    for idx, test in enumerate(task.test):
        if idx < len(predictions):
            pred = predictions[idx]
            if isinstance(pred, list) and pred and isinstance(pred[0], list):
                if isinstance(pred[0][0], list):
                    entries.append({
                        'attempt_1': pred[0],
                        'attempt_2': pred[1] if len(pred) > 1 else pred[0],
                    })
                else:
                    entries.append({'attempt_1': pred, 'attempt_2': pred})
            else:
                entries.append({'attempt_1': pred, 'attempt_2': pred})
        else:
            entries.append({'attempt_1': test.input, 'attempt_2': test.input})
    return entries

def fallback_entry(task_data):
    """Identity fallback: output = input."""
    return [{'attempt_1': t['input'], 'attempt_2': t['input']} for t in task_data['test']]

print('Helper functions defined.')

Helper functions defined.


In [8]:
# Cell 7: Run solver on all tasks
submission = {}
total_start = time.monotonic()
solved_count = 0
n_tasks = len(tasks)

print(f'Starting: {n_tasks} tasks')
print('=' * 60)

for i, (task_id, task_data) in enumerate(tasks.items()):
    elapsed_total = time.monotonic() - total_start
    remaining = MAX_TOTAL_SECONDS - elapsed_total

    # Time budget check
    if remaining < 120:
        print(f'\nTime budget exhausted at task {i+1}/{n_tasks}, filling rest with fallback')
        for tid, tdata in list(tasks.items())[i:]:
            submission[tid] = fallback_entry(tdata)
        break

    task = make_task(task_id, task_data)
    task_start = time.monotonic()

    try:
        solve_dict = solve_task_improved(
            task=task,
            bridge=bridge,
            max_hypotheses=MAX_HYPOTHESES,
            max_iterations=MAX_ITERATIONS,
            config=config,
            analytics=None,
        )
        result = solve_dict['result']
        task_time = time.monotonic() - task_start

        if result.solved:
            solved_count += 1
            status = f'SOLVED [{solve_dict.get("complexity", "?")}]'
        else:
            status = f'{result.best_similarity:.0%}'

        print(f'[{i+1:3d}/{n_tasks}] {task_id}: {status} ({task_time:.1f}s) | total: {elapsed_total/60:.0f}m | solved: {solved_count}')
        submission[task_id] = build_submission_entry(task, result)

    except Exception as e:
        task_time = time.monotonic() - task_start
        print(f'[{i+1:3d}/{n_tasks}] {task_id}: ERROR {e} ({task_time:.1f}s)')
        submission[task_id] = fallback_entry(task_data)

total_time = time.monotonic() - total_start
print('\n' + '=' * 60)
print(f'Done: {solved_count}/{n_tasks} solved ({solved_count/n_tasks:.1%})')
print(f'Total time: {total_time/3600:.1f}h ({total_time:.0f}s)')
print(f'Avg per task: {total_time/n_tasks:.1f}s')
print(f'LLM stats: {bridge.stats()}')

12:58:07 [INFO] [00576224] Phase 0: Attempting transduction...


Starting: 240 tasks


13:13:14 [INFO] Transduction: parsed 6x6 grid, confidence=1.00, valid=True
13:19:08 [INFO] Transduction: parsed 6x6 grid, confidence=1.00, valid=True
13:19:08 [INFO] Training verification passed (2 pairs), predicting test output
13:24:22 [INFO] Transduction: parsed 6x6 grid, confidence=1.00, valid=True
13:24:22 [INFO] [00576224] Transduction succeeded attempt 2 (temp=0.3)
13:24:22 [INFO] [007bbfb7] Phase 0: Attempting transduction...


[  1/240] 00576224: SOLVED [transduced] (1575.3s) | total: 0m | solved: 1


13:24:51 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:25:19 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:25:48 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:26:16 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:26:44 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:26:44 [INFO] Training verification passed (5 pairs), predicting test output
13:27:12 [INFO] Transduction: parsed 9x9 grid, confidence=1.00, valid=True
13:27:12 [INFO] [007bbfb7] Transduction succeeded attempt 1 (temp=0.0)
13:27:12 [INFO] [009d5c81] Phase 0: Attempting transduction...


...

[  3/240] 009d5c81: SOLVED [transduced] (679.1s) | total: 29m | solved: 3


14:10:45 [INFO] Transduction: parsed 10x10 grid, confidence=0.80, valid=True
14:41:39 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
14:44:34 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
14:47:29 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
14:50:23 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
14:53:11 [INFO] Transduction: parsed 19x20 grid, confidence=0.30, valid=False
14:56:05 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
14:59:00 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
15:01:54 [INFO] Transduction: parsed 20x20 grid, confidence=0.80, valid=True
15:01:54 [INFO] Pure D4 vote: 8/8 valid, agreement=88.5%
15:01:54 [INFO] [00d62c1b] Augmented transduction succeeded (pure D4)
15:01:54 [INFO] [00dbd492] Phase 0: Attempting transduction...


[  4/240] 00d62c1b: SOLVED [augmented_transduction] (5003.2s) | total: 40m | solved: 4


15:03:20 [INFO] Transduction: parsed 15x15 grid, confidence=0.80, valid=True
15:17:15 [INFO] Transduction: parsed 15x15 grid, confidence=0.80, valid=True
15:29:50 [INFO] Transduction: parsed 18x9 grid, confidence=0.30, valid=False
15:31:15 [INFO] Transduction: parsed 15x15 grid, confidence=0.80, valid=True
15:31:56 [INFO] Transduction: parsed 9x9 grid, confidence=0.80, valid=True
16:18:11 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:20:45 [INFO] Transduction: parsed 21x20 grid, confidence=0.30, valid=False
16:23:13 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:25:41 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:28:08 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:30:36 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:33:04 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid=False
16:35:31 [INFO] Transduction: parsed 20x20 grid, confidence=0.30, valid

[  5/240] 00dbd492: 78% (35401.4s) | total: 124m | solved: 4

Time budget exhausted at task 6/240, filling rest with fallback

Done: 4/240 solved (1.7%)
Total time: 11.9h (42829s)
Avg per task: 178.5s
LLM stats: LLM Bridge (/kaggle/input/qwen3-8b-unsloth-4bit-quantized): 116 calls, 422666 chars sent, 698136 chars received


In [9]:
# Cell 8: Validate and save submission
# Ensure all tasks are present
missing = set(sample_sub.keys()) - set(submission.keys())
if missing:
    print(f'WARNING: {len(missing)} tasks missing, adding fallback')
    for tid in missing:
        submission[tid] = fallback_entry(tasks[tid])

# Validate format
errors = []
for tid, entries in submission.items():
    if not isinstance(entries, list):
        errors.append(f'{tid}: entries is not a list')
        continue
    for j, entry in enumerate(entries):
        if 'attempt_1' not in entry or 'attempt_2' not in entry:
            errors.append(f'{tid}[{j}]: missing attempt_1 or attempt_2')
        for key in ['attempt_1', 'attempt_2']:
            grid = entry.get(key)
            if not isinstance(grid, list) or not grid or not isinstance(grid[0], list):
                errors.append(f'{tid}[{j}].{key}: not a valid 2D grid')

if errors:
    print(f'Validation errors ({len(errors)}):')
    for e in errors[:10]:
        print(f'  {e}')
else:
    print(f'Validation passed: {len(submission)} tasks, all have attempt_1 + attempt_2')

# Save
OUTPUT_PATH = '/kaggle/working/submission.json' if ON_KAGGLE else 'submission.json'
with open(OUTPUT_PATH, 'w') as f:
    json.dump(submission, f)

file_size = os.path.getsize(OUTPUT_PATH)
print(f'Saved to {OUTPUT_PATH} ({file_size/1024:.1f} KB)')

Validation passed: 240 tasks, all have attempt_1 + attempt_2
Saved to /kaggle/working/submission.json (379.1 KB)
